# Oil Futures Regime-Aware Scenario Forecasting

This notebook is the cleaned GitHub version of the original ARPM-style assignment.

Core workflow:

1. Download weekly WTI, Brent and NVDA prices.
2. Test stationarity of log-prices and log-differences.
3. Fit low-order AR models to weekly log-differences and extract residual innovations.
4. Build half-life flexible probabilities.
5. Generate WTI multi-horizon scenarios with a weighted residual bootstrap.
6. Condition WTI innovation probabilities on macro/risk states: VIX, USD and rates.
7. Validate WTI/Brent joint dynamics through VAR/Johansen cointegration and a VECM hook.
8. Interpret common-vs-spread behavior through a one-factor Kalman/dynamic-factor model.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make local src package importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from oil_futures_regime import (
    download_weekly_close,
    get_weekly_state,
    adf_kpss_table,
    independence_diagnostics,
    fit_ar1_summary,
    fit_ar_orders,
    choose_p_bic_with_lb,
    fit_final_ar,
    extract_ar_params,
    exp_half_life_weights,
    effective_sample_size,
    weighted_mean_cov,
    macro_kernel_weights,
    simulate_ar_bootstrap,
    var_johansen_summary,
    estimate_spread_half_life,
    fit_dynamic_factor_model,
)
from oil_futures_regime.cointegration import fit_vecm
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

## 1. Data

In [ ]:
START = "2013-01-01"
END = "2023-01-01"
TICKERS = {"WTI": "CL=F", "BRENT": "BZ=F", "NVDA": "NVDA"}

px_w = download_weekly_close(TICKERS, START, END, anchor="W-FRI", auto_adjust=False)
logp = np.log(px_w)
dlogp = logp.diff().dropna()

px_w.tail()

## 2. Stationarity and dependence diagnostics

In [ ]:
ar1_rows = []
for col in logp.columns:
    ar1_rows.append({"series": col, "type": "log-price", **fit_ar1_summary(logp[col])})
    ar1_rows.append({"series": col, "type": "log-diff", **fit_ar1_summary(dlogp[col])})

ar1_table = pd.DataFrame(ar1_rows)
ar1_table

In [ ]:
diag_table = pd.DataFrame([
    {"series": col, **independence_diagnostics(dlogp[col], lags=10)}
    for col in dlogp.columns
])
diag_table

In [ ]:
stationarity_table = adf_kpss_table(logp, dlogp)
stationarity_table

## 3. Marginal AR models and innovation extraction

The goal is not to create a complex marginal model. The AR filter removes simple linear autocorrelation before the empirical innovation bootstrap. Remaining volatility clustering is handled non-parametrically through residual bootstrapping and flexible probabilities.

In [ ]:
series_list = ["WTI", "BRENT", "NVDA"]
selection = {}
results = {}
order_tables = {}

for s in series_list:
    tab = fit_ar_orders(dlogp[s], p_max=3, lb_lags=10)
    p_star, passed = choose_p_bic_with_lb(tab, lb_threshold=0.05)
    order_tables[s] = tab
    selection[s] = {"p_star": p_star, "LB_passed": passed}
    results[s] = fit_final_ar(dlogp[s], p_star)

selection

In [ ]:
eps = pd.DataFrame(index=dlogp.index, columns=series_list, dtype=float)
diag = []

for s in series_list:
    e = pd.Series(results[s].resid, index=dlogp[s].dropna().index).reindex(dlogp.index)
    eps[s] = e

    lb_ret = acorr_ljungbox(e.dropna(), lags=[10], return_df=True)["lb_pvalue"].iloc[0]
    lb_abs = acorr_ljungbox(e.dropna().abs(), lags=[10], return_df=True)["lb_pvalue"].iloc[0]
    arch_p = het_arch(e.dropna())[1]

    diag.append({
        "series": s,
        "chosen_p": selection[s]["p_star"],
        "LB_pvalue_resid": float(lb_ret),
        "LB_pvalue_abs_resid": float(lb_abs),
        "ARCH_LM_pvalue": float(arch_p),
    })

diag_resid_table = pd.DataFrame(diag)
diag_resid_table

## 4. Half-life flexible probabilities

In [ ]:
half_life_grid = [13, 26, 52, 78, 104, 156]
ess_table = pd.DataFrame([
    {
        "half_life_weeks": h,
        "ESS": effective_sample_size(exp_half_life_weights(eps.dropna().index, h).values),
    }
    for h in half_life_grid
])

ess_table

In [ ]:
# Select half-life closest to target ESS = 150
target_ess = 150
best_row = ess_table.iloc[(ess_table["ESS"] - target_ess).abs().argmin()]
half_life = int(best_row["half_life_weeks"])

w = exp_half_life_weights(eps.dropna().index, half_life)
eps_joint = eps.dropna().copy()
mu_w, cov_w = weighted_mean_cov(eps_joint, w)

half_life, effective_sample_size(w.values), mu_w, cov_w

## 5. WTI scenario forecast with macro-conditioned weighted residual bootstrap

In [ ]:
# Forecast origin and recent WTI state
t0 = pd.Timestamp("2025-03-05")
start_lookback = "2024-06-01"

wti_recent = download_weekly_close({"WTI": "CL=F"}, start_lookback, (t0 + pd.Timedelta(days=2)).strftime("%Y-%m-%d"), anchor="W-WED", auto_adjust=False)["WTI"]
wti_20 = wti_recent.tail(20)

x20 = np.log(wti_20).dropna()
r20 = x20.diff().dropna().values

c, phi, p = extract_ar_params(results["WTI"])
x0 = float(x20.iloc[-1])
r_init = r20[-p:][::-1] if p > 0 else np.array([])

print("WTI AR order:", p)
print("constant:", c)
print("phi:", phi)
print("x0:", x0)
print("r_init:", r_init)

In [ ]:
# Innovation pool and base time-decay weights
eps_wti = eps["WTI"].dropna()
w_time = w.reindex(eps_wti.index).dropna()
eps_wti = eps_wti.reindex(w_time.index)

# Center residual pool under base weights
eps_wti_centered = eps_wti - np.sum(w_time.values * eps_wti.values)

In [ ]:
# Macro/risk-state conditioning: VIX, USD index, US 10Y yield
state_tickers = ["^VIX", "DX-Y.NYB", "^TNX"]

state_hist = get_weekly_state(state_tickers, start="2012-12-01", end="2023-01-10", anchor="W-WED", auto_adjust=False)
state_hist = state_hist.reindex(eps_wti.index, method="ffill").dropna()

state_full = get_weekly_state(
    state_tickers,
    start="2012-12-01",
    end=(t0 + pd.Timedelta(days=2)).strftime("%Y-%m-%d"),
    anchor="W-WED",
    auto_adjust=False,
)
z0 = state_full.reindex([t0], method="ffill").iloc[0]

idx_common = eps_wti.index.intersection(state_hist.index)
eps_pool = eps_wti_centered.reindex(idx_common)
w_base = w_time.reindex(idx_common)
Z = state_hist.reindex(idx_common)

bw_grid = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0]
target_ess_min = 75

best_bw = None
best_w_total = None
for bw in bw_grid:
    candidate = macro_kernel_weights(Z, z0, w_base, bw=bw)
    if effective_sample_size(candidate) >= target_ess_min:
        best_bw = bw
        best_w_total = candidate
        break

if best_w_total is None:
    best_bw = bw_grid[-1]
    best_w_total = macro_kernel_weights(Z, z0, w_base, bw=best_bw)

w_total = best_w_total

print("Selected macro bandwidth:", best_bw)
print("Macro-conditioned ESS:", effective_sample_size(w_total))

In [ ]:
H = 20
J = 10_000

x_paths, r_paths = simulate_ar_bootstrap(
    x0=x0,
    r_init=r_init,
    c=c,
    phi=phi,
    eps_pool=eps_pool.values,
    prob=w_total,
    horizon=H,
    n_sim=J,
    seed=1,
)

q = [0.05, 0.50, 0.95]
q_paths = np.quantile(x_paths, q, axis=1)

forecast_summary = pd.DataFrame({
    "quantile": q,
    "log_price_H20": q_paths[:, -1],
    "price_H20": np.exp(q_paths[:, -1]),
})
forecast_summary

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(x_paths[-1, :], bins=60, density=True)
plt.title("WTI log-price: 20-week scenario distribution")
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
for i, quantile in enumerate(q):
    plt.plot(np.arange(H + 1), q_paths[i, :], label=f"q{int(quantile * 100)}")
plt.title("WTI log-price forecast quantiles")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## 6. VAR, Johansen cointegration and VECM hook

In [ ]:
Y = logp[["WTI", "BRENT", "NVDA"]].dropna()
cj = var_johansen_summary(Y, var_lags=1, det_order=0, k_ar_diff=1)

print("A1:")
display(cj["A1"])

print("Eigenvalues:", cj["eigenvalues"])
print("Moduli:", cj["moduli"])
print("Johansen trace:", cj["johansen_trace"])
print("Johansen 95% critical values:", cj["johansen_crit_95"])
print("Johansen rank at 95%:", cj["rank95"])

In [ ]:
# Optional VECM fit on WTI/Brent only.
# This turns the Johansen rank decision into an explicit error-correction model.
Y_oil = logp[["WTI", "BRENT"]].dropna()
vecm_res = fit_vecm(Y_oil, coint_rank=1, k_ar_diff=1, deterministic="ci")
print(vecm_res.summary())

## 7. WTI-Brent spread half-life

In [ ]:
spread = (Y_oil["WTI"] - Y_oil["BRENT"]).rename("spread")
spread_info = estimate_spread_half_life(spread)

print("alpha:", spread_info["alpha"])
print("rho:", spread_info["rho"])
print("long-run mean:", spread_info["mu"])
print("half-life weeks:", spread_info["half_life_weeks"])

## 8. One-factor state-space / Kalman interpretation

In [ ]:
df_res = fit_dynamic_factor_model(Y_oil, k_factors=1, factor_order=1, error_order=0)
print(df_res.summary())

In [ ]:
filt_f = df_res.factors.filtered
if isinstance(filt_f, pd.DataFrame):
    factor = filt_f.iloc[:, 0].rename("factor1")
else:
    factor = pd.Series(filt_f[0], index=Y_oil.index, name="factor1")

Z_oil = (Y_oil - Y_oil.mean()) / Y_oil.std()
factor_z = (factor - factor.mean()) / factor.std()

plt.figure(figsize=(12, 4))
plt.plot(Z_oil.index, Z_oil["WTI"], label="WTI standardized")
plt.plot(Z_oil.index, Z_oil["BRENT"], label="Brent standardized")
plt.plot(factor_z.index, factor_z.values, label="Kalman factor standardized", linewidth=2)
plt.title("Kalman-filtered common oil factor")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## 9. Interpretation

The project is strongest as a **market-risk / risk-strat scenario engine**:

- AR filtering removes simple return autocorrelation.
- Flexible probabilities provide a controlled recency weighting scheme.
- Macro-state kernel weights make the residual bootstrap regime-aware.
- Johansen rank 1 and the WTI-Brent spread half-life support a common oil trend plus mean-reverting relative component.
- The dynamic factor model gives a Kalman-filtered common oil factor.

For a stronger **quant research** version, add rolling OOS evaluation, baselines, PIT/CRPS/log-score diagnostics, futures-curve data and transaction-cost-aware decision rules.